In [ ]:
import calendar
import cmocean as cmo
import gc
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import numpy as np
import os

In [ ]:
# Define timestamp version of data to plot
TIMESTAMP_OUT = '01232026_1338'

# Define remaning parameters
HEM = 'sh'
MODEL_DIR = 'cnn-pt'
MODEL_STR = MODEL_DIR.replace('-', '_')

In [ ]:
root_dir = '/data/globus/jbassham/thesis-rough'

In [ ]:
# Load uncertainty (from lr input)
fnam = f'lr-input/{HEM}/{TIMESTAMP_OUT}/test_{HEM}19922020_{TIMESTAMP_OUT}.npz'

data = np.load(os.path.join(root_dir, fnam))
r_test = data['r_test']

In [ ]:
# Use test uncertainty to make land and open ocean mask

land_ocean_mask = np.all(np.isnan(r_test), axis = 0)

In [ ]:
plt.pcolormesh(land_ocean_mask)

In [ ]:
# Expand mask for model outputs
land_ocean_mask_4d = land_ocean_mask[None, None, :, :]

In [ ]:
# Load test split indices (from lr input) 
fnam = f'lr-input/{HEM}/01232026_1338/split_indices_lr_{HEM}19922020_01232026_1338.npz'

data = np.load(os.path.join(root_dir, fnam))
test_idx = data['test_idx']

In [ ]:
# Load coordinates
fnam = f'coordinates/{HEM}/01072026_1643/coord_{HEM}19922020_01072026_1643.npz'

data = np.load(os.path.join(root_dir, fnam))
lat = data['lat']
lon = data['lon']

# Load time in while slicing to test indices
time = data['time'][test_idx]

In [ ]:
# # Load test outputs
fnam = f"model-output/{MODEL_DIR}/{HEM}/{TIMESTAMP_OUT}/preds_{MODEL_STR}_{HEM}19922020_{TIMESTAMP_OUT}.npz"

data = np.load(os.path.join(root_dir, fnam))

y_pred = data['y_pred']
y_true = data['y_true']

print(np.shape(y_pred))
print(np.shape(y_true))
print(np.shape(r_test))

# Apply open ocean and land mask to test outputs
y_pred = np.where(land_ocean_mask_4d, np.nan, y_pred)
y_true = np.where(land_ocean_mask_4d, np.nan, y_true)


In [ ]:
# Get current script directory
script_dir = "/home/jbassham/jack/thesis-rough/notebooks"

# Get root directory (one level above)
root = os.path.abspath(
    os.path.join(
        script_dir,
        '..',
    )
)

# Define function of month plots destination path
PATH_DEST = os.path.abspath(
    os.path.join(
        root,
        'plots',
        'eval_fcn_month',
        MODEL_DIR,
        HEM,
        TIMESTAMP_OUT,
    )
)

# Create the destination directory if it doesn't already exist
os.makedirs(PATH_DEST, exist_ok = True)

In [ ]:
def correlation(pred, true):

    """
    Pearson Correlation
    """

    predbar = np.nanmean(pred, axis = 0) # mean predicted
    truebar = np.nanmean(true, axis = 0) # mean true

    covariance = np.nansum((pred - predbar) * (true - truebar), axis = 0) # covariance between predicted and true
    
    stdpred = np.sqrt(np.nansum((pred - predbar)**2, axis = 0)) # standard deviation predited
    stdtrue = np.sqrt(np.nansum((true - truebar)**2, axis = 0)) # standard deviation true

    correlation = covariance / (stdpred * stdtrue)

    return correlation

In [ ]:
def weighted_correlation(pred, true, r, epsilon = 1e-4):

    """
    Weighted Pearson Correlation referenced from:
    https://www.air.org/sites/default/files/2021-06/Weighted-and-Unweighted-Correlation-Methods-Large-Scale-Educational-Assessment-April-2018.pdf
    
    """

    w = 1 / (r + epsilon)

    def weighted_mean(x, w):
        return np.nansum(w * x, axis = 0) / np.nansum(w, axis = 0)

    predbar = weighted_mean(pred, w) # weighted mean predicted
    truebar = weighted_mean(true, w) # weighted mean true

    weighted_cov = np.nansum(w * (pred - predbar) * (true - truebar), axis = 0) # weighted covariance between predicted and true
    
    weighted_stdpred = np.sqrt(np.nansum(w * (pred - predbar)**2, axis = 0)) # weighted standard deviation predited
    weighted_stdtrue = np.sqrt(np.nansum(w * (true - truebar)**2, axis = 0)) # weighted standard deviation true

    correlation = weighted_cov / (weighted_stdpred * weighted_stdtrue)

    return correlation

In [ ]:
def skill(pred, true, epsilon = 1e-4):
    """
    'Nash-Sutfliffe Efficiency (NSE)'

    - Instable in areas with low variance, error outliers
    """
    # NOTE excluding epsilon = 1e-4 from denominator for now

    mse = np.nanmean((true - pred)**2, axis = 0) # mean square error
    # NOTE above is not equivalent to np.nanvar(true-pred), which excludes bias term
    # MSE = E[(y-x)^2]
    # = (E[y-x])^2 + Var(y-x)
    # = bias^2 + Var(y-x)
    # Can prove the above

    truebar = np.nanmean(true, axis = 0) # mean true

    vartrue = np.nanmean((true - truebar)**2, axis = 0) # variance in true
    # NOTE above is equivalent to np.nanvar()

    skill = 1 - mse / (vartrue + epsilon)

    return skill

In [ ]:
def weighted_skill(pred, true, r, epsilon = 1e-4):
    # NOTE including epsilon = 1e-4 in the weights in case of uncertainty r ~ 0

    w = 1 / (r + epsilon)

    mse = np.nanmean(( w * (true - pred))**2, axis = 0) # mean square error
    # NOTE above is not equivalent to np.nanvar(true-pred), which excludes bias term

    truebar = np.nanmean(true, axis = 0) # mean true

    vartrue = np.nanmean(( w * (true - truebar))**2, axis = 0) # variance in true
    # NOTE above is equivalent to np.nanvar()

    weighted_skill = 1 - mse / (vartrue + epsilon)

    return weighted_skill

In [ ]:
def rmse_skill(pred, true, epsilon = 1e-4):
    """
    'Normalized RMSE skill score', 'RMSE skill score relative to climatology'

    - RMSE of the forecast relative to the RMSE of the referance (std of the observation here)
    - Stable with low-variance outliers

    Used in:
    Hoffman, et. al. (2023)

    Zai, Bitz (2021)

    Mayer, Yang (2023). 'Calibration of derterministic NWP forecsts and its impact on verification'
    https://www.sciencedirect.com/science/article/pii/S0169207022000486?utm_source=chatgpt.com

    """
    # NOTE excluding epsilon = 1e-4 from denominator for now

    mse = np.nanmean((true - pred)**2, axis = 0) # mean square error
    # NOTE above is not equivalent to np.nanvar(true-pred), which excludes bias term
    # MSE = E[(y-x)^2]
    # = (E[y-x])^2 + Var(y-x)
    # = bias^2 + Var(y-x)
    # Can prove the above

    truebar = np.nanmean(true, axis = 0) # mean true

    vartrue = np.nanmean((true - truebar)**2, axis = 0) # variance in true
    # NOTE above is equivalent to np.nanvar()

    skill = 1 - np.sqrt(mse) / np.sqrt((vartrue + epsilon))

    return skill

In [ ]:
def plot_metric2(data, lon, lat, metric, cmap = None, vmin = None, vmax = None):

    # Set longitude bounds for plot (full zonal coverage)
    lon_min = -180
    lon_max = 180

    # Set latitude bounds based on hemisphere
    if HEM == 'sh':
        lat_min = -90
        lat_max = -65
    elif HEM =='nh':
        lat_min = 65
        lat_max = 90

    # Define plot proection based on hempisphere
    if HEM == 'sh':
        projection = ccrs.SouthPolarStereo()
    elif HEM == 'nh':
        projection = ccrs.NorthPolarStereo()

    # Define data-to-plot's coordinate reference system
    # NOTE, used for 'crs' and 'transform' cartopy parameters
    crs = ccrs.PlateCarree()
    
    # Initialize subplots
    fig, axs = plt.subplots(
        nrows = 2,
        ncols = 6,
        figsize = (18,6),
        subplot_kw = {'projection': projection},
        constrained_layout = True
    )

    # Flatten array to single Cartopy GeoAxes for iteration
    axs = axs.flatten()

    # Get month numbers from time variable
    months = (time.astype('datetime64[M]').astype(int) % 12) + 1

    for m in range(12):

        ax = axs[m]

        month_data = data[m]

        # Plot left plot; zonal evaluation
        ax.set_extent([lon_min, lon_max, lat_min, lat_max], crs = crs)
        ax.coastlines()
        # Plot pcolormesh plot
        pcm = ax.pcolormesh(
            lon, lat, month_data,
            transform = crs,
            cmap = cmap, vmin = vmin, vmax = vmax
        )

        # Get month string for plot title
        month_string = calendar.month_name[m + 1]

        ax.set_title(f"{month_string}")

    # Add colorbar
    fig.colorbar(pcm, ax = axs, orientation='vertical', shrink=0.85, pad=0.2)

    # Add title to plot
    fig.suptitle(f"{metric}; {MODEL_STR.upper()}", fontweight = 'bold')

    # Format with tight layout
    fig.tight_layout

    # Define filemane for figure
    fnam = f"{metric}_{MODEL_STR}_{TIMESTAMP_OUT}.png"

    # # Save figure
    # plt.savefig(os.path.join(PATH_DEST, fnam), bbox_inches = 'tight')

    return

In [ ]:
def plot_fcn_month2(y_pred, y_true, time, lon, lat, metric_fcn, metric_str, r, weighted = False, cmap = cmo.cm.balance_r, vmin = -1, vmax = 1):

    """

    Computes and plots the given metric function for each month of the year
    Given inputs:
        y_pred: shaped [nt,2,nlat,nlon]; where y_pred[:,0,:,:] is the u component and y_pred[:,1,:,:] is the v component
        y_true: shaped [nt,2,nlat,nlon]; where y_true[:,0,:,:] is the u component and y_pred[:,1,:,:] is the v component
        lon: longitude
        lat: latitude
        metric_fcn: skill, weighted_skill, correlation, or weighted_correlation functions
        *metric_args and **metric_kwargs: attitional arguments for metric functions (ie: r for weighted)

    """

    # Get shape for spatial dimensions
    nt, _, nlat, nlon = np.shape(y_pred)

    # Define number of month bins
    nm = 12 # number months

    # Initialize empty arrays containing metrics for each month of the year
    u_metric_allyear = np.full((nm, nlat, nlon), np.nan)
    v_metric_allyear = np.full((nm, nlat, nlon), np.nan)

    # Get month numbers from time variable
    months = (time.astype('datetime64[M]').astype(int) % 12) + 1

    # # Create array of month strings for plot titles
    # month_strings = [calendar.month_name[m] for m in range(1,13)]

    for i in range(12):

        month = months == (i + 1)

        # Note: NaN's are time variable, use the uncertainty NaN's to create
        # time variable mask to mask out spurious model outputs (ie: where input
        # for CNN is 0, where imag comp of lr output is 0)

        # Use test uncertainty to make monthly land and open ocean mask
        land_ocean_mask = np.all(np.isnan(r[month,:,:]), axis = 0)

        # Expand to 4d for masking
        land_ocean_mask = land_ocean_mask[None, None, :, :]

        y_pred[month,:,:,:] = np.where(land_ocean_mask, np.nan, y_pred[month,:,:,:])
        y_true[month,:,:,:] = np.where(land_ocean_mask, np.nan, y_true[month,:,:,:])

        # Initialize keyword arguments (r for weighted metrics)
        metric_kwargs = {}

        if weighted:
            metric_kwargs["r"] = r[month]

            u_metric_month = metric_fcn(
                y_pred[month,0,:,:],
                y_true[month,0,:,:],
                **metric_kwargs,
            )
            v_metric_month = metric_fcn(
                y_pred[month,1,:,:],
                y_true[month,1,:,:],
                **metric_kwargs,
            )

        else:
            u_metric_month = metric_fcn(
                y_pred[month,0,:,:],
                y_true[month,0,:,:],
            )
            v_metric_month = metric_fcn(
                y_pred[month,1,:,:],
                y_true[month,1,:,:],
            )

        # Fill current month in year long array
        u_metric_allyear[i, :, :] = u_metric_month
        v_metric_allyear[i, :, :] = v_metric_month

    # Plot the u metrics for each month
    plot_metric2(u_metric_allyear, lon, lat, f"zonal_{metric_str}", cmap = cmap, vmin = vmin, vmax = vmax)

    # Plot the u metrics for each month
    plot_metric2(v_metric_allyear, lon, lat, f"meridional_{metric_str}", cmap = cmap, vmin = vmin, vmax = vmax)

    # Return arrays with metric for each month
    return u_metric_allyear, v_metric_allyear  

In [ ]:
def plot_avg_metric(u_metric, v_metric, metric_str):
    """
    Plots the spatial average of metric for each month in time series
    """

    # Create array of month strings for plot titles
    month_strings = [calendar.month_name[m] for m in range(1,13)]

    if HEM == 'sh':
        hem_str = 'Southern Ocean'
    
    elif HEM == 'nh':
        hem_str = 'Arctic'

    # Average into monthly bins
    u_metric_bar = np.nanmean(u_metric, axis = (1,2))
    v_metric_bar = np.nanmean(v_metric, axis = (1,2))

    plt.plot(u_metric_bar, marker='o', label='zonal')
    plt.plot(v_metric_bar, marker='o', label='meridional')
    plt.xticks(np.arange(12), month_strings, rotation=45)
    plt.ylabel('metric')
    plt.title(f"{hem_str} {metric_str} Spatial Mean; {MODEL_STR.upper()}", fontsize = 14, fontweight = 'bold')
    plt.legend()

    # Define filemane for figure
    fnam = f"{metric_str}year_{MODEL_STR}_{TIMESTAMP_OUT}.png"

    # # Save figure
    # plt.savefig(os.path.join(PATH_DEST, fnam), bbox_inches = 'tight')


    plt.show()


In [ ]:
def plot_median_metric(u_metric, v_metric, metric_str):
    """
    Plots the spatial average of metric for each month in time series
    """

    # Create array of month strings for plot titles
    month_strings = [calendar.month_name[m] for m in range(1,13)]

    if HEM == 'sh':
        hem_str = 'Southern Ocean'
    
    elif HEM == 'nh':
        hem_str = 'Arctic'

    # Average into monthly bins
    u_metric_med = np.nanmedian(u_metric, axis = (1,2))
    v_metric_med = np.nanmedian(v_metric, axis = (1,2))

    plt.plot(u_metric_med, marker='o', label='zonal')
    plt.plot(v_metric_med, marker='o', label='meridional')
    plt.xticks(np.arange(12), month_strings, rotation=45)
    plt.ylabel('metric')
    plt.title(f"{hem_str} {metric_str}; Spatial Median {MODEL_STR.upper()}", fontsize = 14, fontweight = 'bold')
    plt.legend()

    # Define filemane for figure
    fnam = f"{metric_str}year_{MODEL_STR}_{TIMESTAMP_OUT}.png"

    # # Save figure
    # plt.savefig(os.path.join(PATH_DEST, fnam), bbox_inches = 'tight')


    plt.show()

In [ ]:
# # TODO fix dimension persistence

# # NOTE removing last day from time and r for persistence (persistence 1 day short)

# if MODEL_STR == 'ps':
#     # Compute skill as a function of month
#     u_skill_month , v_skill_month = plot_fcn_month(y_pred, y_true, time[:-1], lon, lat, skill, 'skill')

#     # Compute weighted skill as a function of month
#     u_wtdskill_month , v_wtdskill_month = plot_fcn_month(y_pred, y_true, time[:-1], lon, lat, weighted_skill, 'wtd skill', r = r_test[:-1,:,:])

#     # Compute correlation as a function of month
#     u_corr_month_ps , v_corr_month = plot_fcn_month(y_pred, y_true, time[:-1], lon, lat, correlation, 'corr')

#     # Compute weighted correlation as a function of month
#     u_wtdcorr_month , v_wtdcorr_month = plot_fcn_month(y_pred, y_true, time[:-1], lon, lat, weighted_correlation, 'wtd corr', r = r_test[:-1,:,:])

# else: 
#     # Compute skill as a function of month
#     u_skill_month , v_skill_month = plot_fcn_month(y_pred, y_true, time, lon, lat, skill, 'skill')

#     # Compute weighted skill as a function of month
#     u_wtdskill_month , v_wtdskill_month = plot_fcn_month(y_pred, y_true, time, lon, lat, weighted_skill, 'wtd skill', r = r_test[:-1,:,:])

#     # Compute correlation as a function of month
#     u_corr_month_ps , v_corr_month = plot_fcn_month(y_pred, y_true, time, lon, lat, correlation, 'corr')

#     # Compute weighted correlation as a function of month
#     u_wtdcorr_month , v_wtdcorr_month = plot_fcn_month(y_pred, y_true, time, lon, lat, weighted_correlation, 'wtd corr', r = r_test[:-1,:,:])



In [ ]:
# TODO fix dimension persistence

# NOTE removing last day from time and r for persistence (persistence 1 day short)

if MODEL_STR == 'ps':
    # Compute skill as a function of month
    u_skill_month , v_skill_month = plot_fcn_month2(y_pred, y_true, time[:-1], lon, lat, skill, 'skill', r = r_test[:-1,:,:])

    # Compute rmse skill as a function of month
    u_rmseskill_month , v_rmseskill_month = plot_fcn_month2(y_pred, y_true, time[:-1], lon, lat, rmse_skill, 'rmse_skill', r = r_test[:-1,:,:])

    # Compute weighted skill as a function of month
    u_wtdskill_month , v_wtdskill_month = plot_fcn_month2(y_pred, y_true, time[:-1], lon, lat, weighted_skill, 'wtd skill', r = r_test[:-1,:,:], weighted = True)

    # Compute correlation as a function of month
    u_corr_month , v_corr_month = plot_fcn_month2(y_pred, y_true, time[:-1], lon, lat, correlation, 'corr', r = r_test[:-1,:,:])

    # Compute weighted correlation as a function of month
    u_wtdcorr_month , v_wtdcorr_month = plot_fcn_month2(y_pred, y_true, time[:-1], lon, lat, weighted_correlation, 'wtd corr', r = r_test[:-1,:,:], weighted = True)

else: 
    # Compute skill as a function of month
    u_skill_month , v_skill_month = plot_fcn_month2(y_pred, y_true, time, lon, lat, skill, 'skill', r = r_test)

    # Compute rmse skill as a function of month
    u_rmseskill_month , v_rmseskill_month = plot_fcn_month2(y_pred, y_true, time, lon, lat, rmse_skill, 'rmse_skill', r = r_test)

    # Compute weighted skill as a function of month
    u_wtdskill_month , v_wtdskill_month = plot_fcn_month2(y_pred, y_true, time, lon, lat, weighted_skill, 'wtd skill', r = r_test, weighted = True)

    # Compute correlation as a function of month
    u_corr_month , v_corr_month = plot_fcn_month2(y_pred, y_true, time, lon, lat, correlation, 'corr', r = r_test)

    # Compute weighted correlation as a function of month
    u_wtdcorr_month , v_wtdcorr_month = plot_fcn_month2(y_pred, y_true, time, lon, lat, weighted_correlation, 'wtd corr', r = r_test, weighted = True)

In [ ]:
def pdf_monthly(u_data, v_data, metric_str = None):

    # Initialize subplots for 12 months
    fig, axs = plt.subplots(
        nrows = 2,
        ncols = 6,
        figsize = (18,6),
        constrained_layout = True
    )

    # Flatten axs array for iteration
    axs = axs.flatten()

    # Define number of bins
    n_bins = 30

    # Iterate through months
    for m in range(12):
        
        # Get axis for current month
        ax = axs[m]

        # Plot u histogram with transparency
        ax.hist(u_data[m,:,:].flatten(), alpha = 0.85, bins = n_bins, label = 'zonal')

        # Plot v histogram with transparency
        ax.hist(v_data[m,:,:].flatten(), alpha = 0.85, bins = n_bins, label = 'meridional')

        ax.set_xlim(-800, 0)

        ax.set_ylim(0, 10)

        # Get month string for plot title
        month_string = calendar.month_name[m + 1]

        ax.set_title(f"{month_string}")

    # Get axis handles and lables from one axs
    h, l = axs[0].get_legend_handles_labels()

    fig.legend(h, l)

    # Define region string for plot title
    if HEM == 'sh':
        hem_str = 'Southern Ocean'
    
    else:
        hem_str = 'Arctic'

    fig.suptitle(f"{hem_str} {metric_str}", fontweight = 'bold')

    plt.show()



In [ ]:
# Compute mean skill each month
pdf_monthly(u_skill_month , v_skill_month, 'skill')

# Compute mean rmse skill each month
pdf_monthly(u_rmseskill_month , v_rmseskill_month, 'rmse_skill')

# Compute mean weighted skill each month
pdf_monthly(u_wtdskill_month , v_wtdskill_month, 'wtd skill')

# Compute mean weighted skill each month
pdf_monthly(u_corr_month , v_corr_month, 'corr')

# Compute mean weighted skill each month
pdf_monthly(u_wtdcorr_month , v_wtdcorr_month, 'wtd corr')

In [ ]:
def plot_outliers(u_data, v_data, metric_str, cmap, vmin = None, vmax = None, limit = -1):

    u_data_outlier = np.where(u_data < limit, u_data, np.nan)

    v_data_outlier = np.where(v_data < limit, v_data, np.nan)

    plot_metric2(u_data_outlier, lon, lat, f"Zonal Outliers {metric_str}", cmap = cmap, vmin = vmin, vmax = vmax)

    plot_metric2(u_data_outlier, lon, lat, f"Meridional Outliers {metric_str}", cmap = cmap, vmin = vmin, vmax = vmax)

    return

In [ ]:
plot_outliers(u_skill_month, v_skill_month, 'skill', cmo.cm.thermal)

plot_outliers(u_rmseskill_month , v_rmseskill_month, 'rmse_skill', cmo.cm.thermal)

plot_outliers(u_wtdskill_month , v_wtdskill_month, 'wtd skill', cmo.cm.thermal)

plot_outliers(u_corr_month , v_corr_month, 'corr', cmo.cm.thermal)

plot_outliers(u_wtdcorr_month , v_wtdcorr_month, 'wtd corr', cmo.cm.thermal)

In [ ]:
limit = -100

plot_outliers(u_skill_month, v_skill_month, 'skill', cmo.cm.thermal, limit = limit)

plot_outliers(u_rmseskill_month , v_rmseskill_month, 'rmse_skill', cmo.cm.thermal, limit = limit)

plot_outliers(u_wtdskill_month , v_wtdskill_month, 'wtd skill', cmo.cm.thermal, limit = limit)

plot_outliers(u_corr_month , v_corr_month, 'corr', cmo.cm.thermal, limit = limit)

plot_outliers(u_wtdcorr_month , v_wtdcorr_month, 'wtd corr', cmo.cm.thermal, limit = limit)

In [ ]:
def mean_error(pred, true):

    error = pred - true

    mean_error = np.nanmean(pred - true, axis = 0)

    return mean_error

In [ ]:
def max_error(pred, true):

    error = pred - true

    max_error = np.nanmax(pred - true, axis = 0)

    return max_error

In [ ]:
u_me_month, v_me_month = plot_fcn_month2(
    y_pred, y_true, 
    time, lon, lat, 
    mean_error, 'mean_error', 
    r = r_test, 
    cmap = cmo.cm.balance_r, vmin = None, vmax = None
    )


In [ ]:
u_maxe_month, v_maxe_month = plot_fcn_month2(
    y_pred, y_true, 
    time, lon, lat, 
    max_error, 'max_error', 
    r = r_test, 
    cmap = cmo.cm.thermal, vmin = None, vmax = None
    )

In [ ]:
# Compute mean skill each month
plot_avg_metric(u_skill_month , v_skill_month, 'skill')

# Compute mean rmse skill each month
plot_avg_metric(u_rmseskill_month , v_rmseskill_month, 'rmse_skill')

# Compute mean weighted skill each month
plot_avg_metric(u_wtdskill_month , v_wtdskill_month, 'wtd skill')

# Compute mean weighted skill each month
plot_avg_metric(u_corr_month , v_corr_month, 'corr')

# Compute mean weighted skill each month
plot_avg_metric(u_wtdcorr_month , v_wtdcorr_month, 'wtd corr')



In [ ]:
# Compute median skill each month

plot_median_metric(u_skill_month , v_skill_month, 'skill')

plot_median_metric(u_rmseskill_month , v_rmseskill_month, 'rmse_skill')

plot_median_metric(u_wtdskill_month , v_wtdskill_month, 'wtd skill')

plot_median_metric(u_corr_month , v_corr_month, 'corr')

plot_median_metric(u_wtdcorr_month , v_wtdcorr_month, 'wtd corr')